# Steam video games platform analysis

This project analyzes **the video game market on the Steam marketplace**, with the goal of understanding factors which affect a game's popularity and sales performance.

The analysis is conducted using exploratory data analysis and visualizations on a large dataset of of approximately 55,600 video games.

We will go through the following steps using PySpark on Databricks:
- Ingesting data
- Cleaning data for analysis
- Performing exploratory data analysis and creating visualisations

Different levels of analysis will be carried out:

**Analysis at the "macro" level**  
Which publisher has released the most games on Steam?  
What are the best rated games?  
Are there years with more releases? Were there more or fewer game releases during the Covid, for example?  
How are the prizes distributed? Are there many games with a discount?  
What are the most represented languages?  
Are there many games prohibited for children under 16/18?  

**Genres analysis**  
What are the most represented genres?  
Are there any genres that have a better positive/negative review ratio?  
Do some publishers have favorite genres?  
What are the most lucrative genres?  

**Platform analysis**  
Are most games available on Windows/Mac/Linux?  
Do certain genres tend to be preferentially available on certain platforms?

In [0]:
# Import packages
import datetime
import pandas as pd
from pyspark.sql.functions import abs, col, concat,count, desc, explode, isnan, lit, mean, regexp_extract, regexp_replace, round, row_number, split, stddev, sum, to_date, trim, when, year
from pyspark.sql.types import StringType, ArrayType, StructType, StructField
from pyspark.sql.window import Window
import builtins
pd.set_option('display.max_columns', None)

# 1. Importing data
We load the dataset: a JSON file stored in an AWS S3 bucket, which contains semi-structured data.

In [0]:
filepath = "s3://full-stack-bigdata-datasets/Big_Data/Project_Steam/steam_game_output.json"

df_raw = spark.read.format('json')\
             .option('header', 'true')\
             .load(filepath)

In [0]:
df_raw.printSchema()

root
 |-- data: struct (nullable = true)
 |    |-- appid: long (nullable = true)
 |    |-- categories: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- ccu: long (nullable = true)
 |    |-- developer: string (nullable = true)
 |    |-- discount: string (nullable = true)
 |    |-- genre: string (nullable = true)
 |    |-- header_image: string (nullable = true)
 |    |-- initialprice: string (nullable = true)
 |    |-- languages: string (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- negative: long (nullable = true)
 |    |-- owners: string (nullable = true)
 |    |-- platforms: struct (nullable = true)
 |    |    |-- linux: boolean (nullable = true)
 |    |    |-- mac: boolean (nullable = true)
 |    |    |-- windows: boolean (nullable = true)
 |    |-- positive: long (nullable = true)
 |    |-- price: string (nullable = true)
 |    |-- publisher: string (nullable = true)
 |    |-- release_date: string (nullable = true)
 |    |-

In [0]:
df_raw.limit(5).toPandas()

,data,id
0,"{'appid': 10, 'categories': ['Multi-player', '...",10
1,"{'appid': 1000000, 'categories': ['Single-play...",1000000
2,"{'appid': 1000010, 'categories': ['Single-play...",1000010
3,"{'appid': 1000030, 'categories': ['Multi-playe...",1000030
4,"{'appid': 1000040, 'categories': ['Single-play...",1000040


The `id` variable seems irrelevant for the analysis. We can focus on the `data` variable, which contains also the product id, called `appid`.

In [0]:
df = df_raw.select("data.*")
df.printSchema()

root
 |-- appid: long (nullable = true)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- ccu: long (nullable = true)
 |-- developer: string (nullable = true)
 |-- discount: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- header_image: string (nullable = true)
 |-- initialprice: string (nullable = true)
 |-- languages: string (nullable = true)
 |-- name: string (nullable = true)
 |-- negative: long (nullable = true)
 |-- owners: string (nullable = true)
 |-- platforms: struct (nullable = true)
 |    |-- linux: boolean (nullable = true)
 |    |-- mac: boolean (nullable = true)
 |    |-- windows: boolean (nullable = true)
 |-- positive: long (nullable = true)
 |-- price: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- required_age: string (nullable = true)
 |-- short_description: string (nullable = true)
 |-- tags: struct (nullable = true)
 |    |-- 1980s: lon

In [0]:
df.limit(5).toPandas()

,appid,categories,ccu,developer,discount,genre,header_image,initialprice,languages,name,negative,owners,platforms,positive,price,publisher,release_date,required_age,short_description,tags,type,website
0,10,"[Multi-player, Valve Anti-Cheat enabled, Onlin...",13990,Valve,0,Action,https://cdn.akamai.steamstatic.com/steam/apps/...,999,"English, French, German, Italian, Spanish - Sp...",Counter-Strike,5199,"10,000,000 .. 20,000,000","{'linux': True, 'mac': True, 'windows': True}",201215,999,Valve,2000/11/1,0,Play the world's number 1 online action game. ...,"{'1980s': 266.0, '1990's': 1191.0, '2.5D': Non...",game,
1,1000000,"[Single-player, Partial Controller Support, St...",0,IndigoBlue Game Studio,0,"Action, Adventure, Indie",https://cdn.akamai.steamstatic.com/steam/apps/...,999,"English, Korean, Simplified Chinese",ASCENXION,5,"0 .. 20,000","{'linux': False, 'mac': False, 'windows': True}",27,999,PsychoFlux Entertainment,2021/05/14,0,ASCENXION is a 2D shoot 'em up game where you ...,"{'1980s': None, '1990's': None, '2.5D': None, ...",game,
2,1000010,"[Single-player, Partial Controller Support, St...",99,NEXT Studios,70,"Adventure, Indie, RPG, Strategy",https://cdn.akamai.steamstatic.com/steam/apps/...,1999,"Simplified Chinese, English, Japanese, Traditi...",Crown Trick,646,"200,000 .. 500,000","{'linux': False, 'mac': False, 'windows': True}",4032,599,"Team17, NEXT Studios",2020/10/16,0,"Enter a labyrinth that moves as you move, wher...","{'1980s': None, '1990's': None, '2.5D': None, ...",game,
3,1000030,"[Multi-player, Single-player, Co-op, Steam Ach...",76,Vertigo Gaming Inc.,0,"Action, Indie, Simulation, Strategy",https://cdn.akamai.steamstatic.com/steam/apps/...,1999,English,"Cook, Serve, Delicious! 3?!",115,"100,000 .. 200,000","{'linux': False, 'mac': True, 'windows': True}",1575,1999,Vertigo Gaming Inc.,2020/10/14,0,"Cook, serve and manage your food truck as you ...","{'1980s': None, '1990's': None, '2.5D': None, ...",game,http://www.cookservedelicious.com
4,1000040,[Single-player],0,DoubleC Games,0,"Action, Casual, Indie, Simulation",https://cdn.akamai.steamstatic.com/steam/apps/...,199,Simplified Chinese,细胞战争,1,"0 .. 20,000","{'linux': False, 'mac': False, 'windows': True}",0,199,DoubleC Games,2019/03/30,0,这是一款打击感十足的细胞主题游戏！操作简单但活下去却不简单，“你”作为侵入人体的细菌病毒，通...,"{'1980s': None, '1990's': None, '2.5D': None, ...",game,


# 2. Data cleaning

## 2.1. First statistics

In [0]:
# Basic statistics

print("Shape of the df :", (df.count(), len(df.columns)))

nb_game = df.select('appid').distinct().count()
nb_publisher = df.select('publisher').distinct().count()
print(f"The dataset contains information for {nb_game} games and {nb_publisher} publishers.")

Shape of the df : (55691, 22)
The dataset contains information for 55691 games and 29966 publishers.


In [0]:
# Check for duplicates
if df.count() > df.dropDuplicates().count():
    raise Exception('The dataset has duplicates.')
else:
  print('The dataset has no duplicates.')

The dataset has no duplicates.


In [0]:
# Check for missing values
missing_counts = df.select([
    sum(col(c).isNull().cast("int")).alias(c) for c in df.columns
])

missing_counts.toPandas().T.rename(columns={0: 'count_missing_values'})

,count_missing_values
appid,0
categories,0
ccu,0
developer,0
discount,0
genre,0
header_image,0
initialprice,0
languages,0
name,0


## 2.2. Creating the release year

In [0]:
df.select(col("release_date")).distinct().limit(5).show()

+------------+
|release_date|
+------------+
|  2019/08/19|
|   2009/03/4|
|  2019/01/26|
|  2020/06/24|
|  2019/06/12|
+------------+



In [0]:
df.select('release_date').summary().show()

+-------+------------+
|summary|release_date|
+-------+------------+
|  count|       55691|
|   mean|        NULL|
| stddev|        NULL|
|    min|            |
|    25%|        NULL|
|    50%|        NULL|
|    75%|        NULL|
|    max|   2022/11/7|
+-------+------------+



In [0]:
# Check for missing or empty values
nb_na_empty = df.filter(
    df["release_date"].isNull() | 
    (trim(col("release_date")) == "")
    ).count()

print(f"The dataset has {nb_na_empty} missing or empty values for release dates")

The dataset has 99 missing or empty values for release dates


In [0]:
# Handle release dates written as "yyyy/MM" (missing day): add a fictive day (01)
df = df.withColumn(
    "release_date_clean",
    when(
        col("release_date").rlike(r"^\d{4}/\d{2}$"),
        concat(col("release_date"), lit("/01"))
    ).otherwise(col("release_date"))
)

# Replace the empty values by missing values and convert it to Date type
df = df.withColumn(
    "release_date_clean",
    to_date(
        when(
            trim(col("release_date_clean")) == "", 
            None
        ).otherwise(col("release_date_clean")),
        "yyyy/MM/d"
    )
)


In [0]:
df.select("release_date_clean").show(10)

+------------------+
|release_date_clean|
+------------------+
|        2000-11-01|
|        2021-05-14|
|        2020-10-16|
|        2020-10-14|
|        2019-03-30|
|        2019-06-24|
|        2019-01-24|
|        2019-04-08|
|        2019-01-06|
|        2021-09-09|
+------------------+
only showing top 10 rows


In [0]:
# Create the release year
df = df.withColumn("release_year", year("release_date_clean"))
df.select(col("release_year")).distinct().limit(10).show()

+------------+
|release_year|
+------------+
|        2018|
|        NULL|
|        2000|
|        2013|
|        2005|
|        2009|
|        2008|
|        2019|
|        2021|
|        2020|
+------------+



In [0]:
df = df.drop('release_date_clean')

## 2.3. Changing types and rescaling variables
The initial prices, the actual prices and the percentage of discount will be converted to floats.  
The prices are expressed in cents, so we convert them into USD by dividing the values by 100.

In [0]:
data_types = {
    'initialprice': 'float',
    'price': 'float',
    'discount': 'float'
  }

for column_name, data_type in data_types.items():
    df = df\
        .withColumn(column_name, col(column_name).cast(data_type))

In [0]:
df.select("initialprice", "price", "discount").show(5)

+------------+------+--------+
|initialprice| price|discount|
+------------+------+--------+
|       999.0| 999.0|     0.0|
|       999.0| 999.0|     0.0|
|      1999.0| 599.0|    70.0|
|      1999.0|1999.0|     0.0|
|       199.0| 199.0|     0.0|
+------------+------+--------+
only showing top 5 rows


In [0]:
for column_name in ['initialprice', 'price']:
    df = df.withColumn(column_name, col(column_name)/100)

df.select("initialprice", "price", "discount").show(5)

+------------+-----+--------+
|initialprice|price|discount|
+------------+-----+--------+
|        9.99| 9.99|     0.0|
|        9.99| 9.99|     0.0|
|       19.99| 5.99|    70.0|
|       19.99|19.99|     0.0|
|        1.99| 1.99|     0.0|
+------------+-----+--------+
only showing top 5 rows


## 2.4. Creating the *revenue* variable
For each video game, we compute the revenue by multiplying the price with the number of owners.
The number of owners is used as an estimation of the number of purchasers. Their exact number is unknown; instead the database provides intervals. Given the lack of information, we will use an approximation which can be questionable: we will approximate the number of owners by the midpoint of the intervals.

In [0]:
# Display the intervals for the number of owners and the associated number of video games
display(df.select('appid', 'owners')\
    .dropDuplicates()\
    .groupBy('owners')\
    .count()\
    .orderBy(desc('count')))

owners,count
"0 .. 20,000",38072
"20,000 .. 50,000",7285
"50,000 .. 100,000",3695
"100,000 .. 200,000",2519
"200,000 .. 500,000",2162
"500,000 .. 1,000,000",933
"1,000,000 .. 2,000,000",526
"2,000,000 .. 5,000,000",335
"5,000,000 .. 10,000,000",97
"10,000,000 .. 20,000,000",41


Most of the video games have owners between 0 and 20 000 units.

In [0]:
# Create the midpoint of the intervals
# Remove the commas from the owners field, split on " .. ", extract min and max values and create the midpoint value
df = df.withColumn('owners_clean', regexp_replace(col('owners'), ',', ''))\
    .withColumn('owners_clean', split(col('owners_clean'), ' \\.\\. '))\
    .withColumn('owners_min', col('owners_clean')[0].cast('int')) \
    .withColumn('owners_max', col('owners_clean')[1].cast('int'))\
    .withColumn('owners_midpoint', (col('owners_min') + col('owners_max')) / 2)

# Create the revenue
df = df.withColumn('revenue', col('price') * col('owners_midpoint'))

In [0]:
df = df.drop('owners_clean')

## 2.5. Creating the proportion of positive ratings

In [0]:
df = df\
    .withColumn('total_reviews', col('positive')+col('negative'))\
    .withColumn(
        'score', 
        when(col('total_reviews') != 0, col('positive')/col('total_reviews')).otherwise(None)
    )

The proportion of positive ratings is only defined when the total reviews are strictly positive.

## 2.6. Flatten the nested structure
As the dataset is semi-structured with a nested schema, we use Pyspark's `getField()` and `explode()` methods to flatten it.

### 2.6.1. Fixing the columns: expanding StructType fields into separate columns

We will focus on the *platforms* variable which will be used further in the analysis.

In [0]:
# Creating new columns for subfields and deleting the original nested field
platform_fields = df.select("platforms.*").columns
for field in platform_fields:
    field_clean = field.lower()
    df = df.withColumn(f"platform_{field_clean}", col("platforms").getField(field))
df = df.drop("platforms")

In [0]:
display(df.limit(5))

appid,categories,ccu,developer,discount,genre,header_image,initialprice,languages,name,negative,owners,positive,price,publisher,release_date,required_age,short_description,tags,type,website,release_year,owners_min,owners_max,owners_midpoint,revenue,total_reviews,score,platform_linux,platform_mac,platform_windows
10,"List(Multi-player, Valve Anti-Cheat enabled, Online PvP, Shared/Split Screen PvP, PvP)",13990,Valve,0.0,Action,https://cdn.akamai.steamstatic.com/steam/apps/10/header.jpg?t=1666823513,9.99,"English, French, German, Italian, Spanish - Spain, Simplified Chinese, Traditional Chinese, Korean",Counter-Strike,5199,"10,000,000 .. 20,000,000",201215,9.99,Valve,2000/11/1,0,Play the world's number 1 online action game. Engage in an incredibly realistic brand of terrorist warfare in this wildly popular team-based game. Ally with teammates to complete strategic missions. Take out enemy sites. Rescue hostages. Your role affects your team's success. Your team's success affects your role.,"List(266, 1191, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 5426, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 227, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 2784, null, null, null, null, null, null, null, null, null, null, null, null, 1607, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 4831, null, null, null, null, null, null, null, null, null, 1707, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 632, null, null, null, null, null, null, null, null, null, null, null, 3392, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 131, null, null, 769, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 881, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 289, null, null, null, 3353, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 614, null, null, null, null, null, null, 304, null, null, null, 1344, null, null, 1864, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 1192)",game,,2000,10000000,20000000,1.5E7,1.4985E8,206414,0.9748127549487923,true,true,true
1000000,"List(Single-player, Partial Controller Support, Steam Achievements, Steam Cloud)",0,IndigoBlue Game Studio,0.0,"Action, Adventure, Indie",https://cdn.akamai.steamstatic.com/steam/apps/1000000/header.jpg?t=1655723048,9.99,"English, Korean, Simplified Chinese",AS

### 2.6.2. Fixing the rows: exploding ArrayType fields into separate rows
For the analysis, we need to fix the *languages* and *genre* variables, by converting them to Array before using the `explode` function.

#### Languages

In [0]:
display(df.select(col("languages")).distinct().limit(5))

languages
"English, French, Simplified Chinese, Greek, Hungarian, Italian, Spanish - Latin America, Turkish, Spanish - Spain, Portuguese - Brazil, Russian"
"English, French, German, Spanish - Spain, Portuguese - Portugal, Portuguese - Brazil, Russian, Spanish - Latin America, Traditional Chinese"
"English, Spanish - Spain, French, German, Dutch, Japanese, Portuguese - Portugal, Portuguese - Brazil, Russian, Simplified Chinese"
"English, Simplified Chinese, French, German, Russian, Polish, Japanese"
"English, French, Italian, Spanish - Spain, Japanese, Russian, Simplified Chinese, Spanish - Latin America, Portuguese - Portugal, Portuguese - Brazil, German"


In [0]:
display(df.select(col('genre')).distinct().limit(5))

genre
"Action, Free to Play, Indie, Early Access"
"Action, Adventure, Casual, Indie, Racing, Simulation"
"Action, Casual, Free to Play, Indie"
Adventure
"Action, Casual, Racing, Simulation"


In [0]:
array_field_list = ['genre', 'languages']

# Convert "languages" and "genre" field from string to array
# Use a regex in the split function to split on commas or newline characters
for var in array_field_list:
    df = df.withColumn(var, split(col(var), r",|\n"))

In [0]:
df.select(col("languages")).printSchema()
df.select(col('genre')).printSchema()

root
 |-- languages: array (nullable = true)
 |    |-- element: string (containsNull = false)

root
 |-- genre: array (nullable = true)
 |    |-- element: string (containsNull = false)



In [0]:
# Exploding Arrays into rows and trim the spaces from both ends
array_field_list = ['genre', 'languages']
for var in array_field_list:
    df = df.withColumn(var, explode(col(var)))\
            .withColumn(var, trim(col(var)))

# Delete the "[b]*[/b]" string which appears in several values of the languages field
df = df.withColumn(
    'languages',
    regexp_replace(col('languages'), r"\[b\]\*\[/b\]", "")
)

# Print distinct values of languages and genre
print("Distinct values of languages:")
display(df.select(col('languages')).distinct())
print("Distinct values of genre:")
display(df.select(col('genre')).distinct())

Distinct values of languages:


languages
Lithuanian
Luxembourgish
Dari
(all with full audio support)
French
Portuguese - Brazil
Turkish
Dutch
Icelandic
Bulgarian


Distinct values of genre:


genre
Adventure
Violent
Racing
Nudity
Indie
Strategy
Education
""
Casual
RPG


In [0]:
# Replace missing values or empty values of genre by "Unknown"
df = df.withColumn(
    'genre',
    when((col('genre').isNull()) | (col('genre') == ""), "Unknown").otherwise(col('genre'))
)

# 3. Exploratory data analysis

## Which publisher has released the most games on Steam?

In [0]:
df_publisher = df\
    .select('appid', 'publisher')\
    .dropDuplicates()\
    .groupBy('publisher')\
    .count()\
    .orderBy(desc('count'))

In [0]:
# Top 5 publishers
display(df_publisher.limit(5))

publisher,count
Big Fish Games,422
8floor,202
SEGA,165
Strategy First,151
Square Enix,141


Databricks visualization. Run in Databricks to view.

Big Fish Games is the publisher with the most releases on Steam (422 video games released out of a total of 55691). The second best publisher is 8floor with half of Big Fish Games' number of releases.

## Distribution of publishers by number of released games

In [0]:
df_publisher_by_nbgame = df_publisher.withColumnRenamed("count", "nb_games")\
	.groupBy("nb_games")\
    .count()\
    .withColumnRenamed("count", "nb_publishers")\
    .orderBy("nb_games")\
    .withColumn(
        "prct_publishers",
        round((col("nb_publishers")/nb_publisher), 2)
    ) \
    .orderBy("nb_games")


In [0]:
display(df_publisher_by_nbgame)

nb_games,nb_publishers,prct_publishers
1,23183,0.77
2,3595,0.12
3,1258,0.04
4,585,0.02
5,341,0.01
6,195,0.01
7,117,0.0
8,92,0.0
9,74,0.0
10,58,0.0


Databricks visualization. Run in Databricks to view.

The distribution of publishers by number of released games shows a highly fragmented market. Most publishers (89%) have released only one or two games, while a few publishers have released hundreds of games, indicating a long-tail distribution.


## Which are the best rated games?

We interpret the rating as the number of positive reviews.
We also compute additional information: the proportion of positive ratings.

In [0]:
df\
    .select('appid', 'name', 'publisher', 'positive', 'negative', 'total_reviews', 'score')\
    .filter(col('total_reviews') > 0)\
    .dropDuplicates()\
    .orderBy(desc('positive'), desc('score'))\
    .limit(5)\
    .toPandas()

,appid,name,publisher,positive,negative,total_reviews,score
0,730,Counter-Strike: Global Offensive,Valve,5943345,787093,6730438,0.883055
1,570,Dota 2,Valve,1534895,317916,1852811,0.828414
2,271590,Grand Theft Auto V,Rockstar Games,1229265,213379,1442644,0.852092
3,578080,PUBG: BATTLEGROUNDS,"KRAFTON, Inc.",1185361,908515,2093876,0.566108
4,105600,Terraria,Re-Logic,1014711,22380,1037091,0.978420


The top 5 games with the most positive reviews are:
- Counter-Strike: Global Offensive
- Dota 2
- Grand Theft Auto V
- PUBG: BATTLEGROUNDS
- Terraria

Counter-Strike, Dota 2, Grand Theft Auto V and Terraria display positive reviews, both in absolute terms and in proportion, indicating a strong user satisfaction. In contrast, PUBG: BATTLEGROUNDS, despite its 1.2 million positive reviews, shows a relatively high number of negative reviews, which leads to a low proportion of positive ratings (56.6%).

## Are there years with more releases? Were there more or fewer game releases during the Covid?

In [0]:
# Compute the annual number of releases
df_release_year = df\
    .select('appid', 'release_year')\
    .dropDuplicates()\
    .filter(col('release_year').isNotNull())\
    .groupBy('release_year')\
    .count()\
    .orderBy(desc('count'))

In [0]:
display(df_release_year)

release_year,count
2021,8823
2020,8305
2018,7678
2022,7455
2019,6968
2017,6017
2016,4185
2015,2576
2014,1557
2013,471


Databricks visualization. Run in Databricks to view.

The Covid years are the ones with the most releases.
2020, 2021 and 2022 are in the top 5 years with the most releases.

## How are the prices distributed?

In [0]:
# Descriptive statistics of actual price
df_price = df.select('appid','price')\
    .dropDuplicates()

df_price.select('price').summary().show()

+-------+-----------------+
|summary|            price|
+-------+-----------------+
|  count|            55691|
|   mean|7.732849832104516|
| stddev|10.93134582723452|
|    min|              0.0|
|    25%|             1.32|
|    50%|             4.99|
|    75%|             9.99|
|    max|            999.0|
+-------+-----------------+



The price distribution is left-skewed, as the mean price (7,73 USD) is greater than the median price (4,99 USD).
50% the prices are between 1,34 USD and 9,99 USD.
Extreme values are located on the right side of the distribution (ex: 999 USD).

In order to build a meaningful histogram, we want to exclude outliers from the scope of the visualisation. Thus we create a trimmed dataset, excluding outliers identified using the z-score method. This method measures how far an observation is from the mean, expressed in terms of standard deviations. Typically a z-score higher than 2 or 3 is considered as an outlier. Here we will fix the threshold at 3.

In [0]:
# Create the trimmed dataset, without price outliers
price_mean = df_price.select(mean(col('price'))).collect()[0][0]
price_std = df_price.select(stddev(col('price'))).collect()[0][0]
df_trimmed = df_price.withColumn('z_score', (col('price') - price_mean) / price_std) \
               .filter(abs(col('z_score')) <= 3)

In [0]:
display(df_trimmed[['price']])

price
0.0
4.99
19.99
0.0
9.99
3.99
3.99
15.99
0.0
0.99


Databricks visualization. Run in Databricks to view.

In [0]:
# Free video games
nb_free = df_price.filter(col('price') == 0).count()
nb_total = df_price.count()
print(f"{nb_free} video games are free, accounting for {builtins.round(nb_free/nb_total, 1)} % of the total number of video games.")


7780 video games are free, accounting for 0.1 % of the total number of video games.


## Are there many games with a discount?

In [0]:
# Create a discount indicator : 'discount' if the price is discounted, 'no discount' otherwise
df_discount = df\
    .select('appid', 'discount')\
    .dropDuplicates()\
    .withColumn(
        "discount_indicator",
        when(
            col("discount") > 0,
            "discount"
        ).otherwise("no discount")
    )\
    .groupBy('discount_indicator')\
    .count()

In [0]:
display(df_discount)

discount_indicator,count
no discount,53173
discount,2518


Databricks visualization. Run in Databricks to view.

4.52% of the games are discounted (2518 games out of a total of 55 691).

## Which are the most represented languages?

In [0]:
df_language = df\
    .select('appid', 'languages')\
    .dropDuplicates()

count_languages = df_language.count()

# define a window, necessary to compute the cumulative count and proportion
w = Window.orderBy(desc("count"))

df_language_count = df_language\
    .groupBy('languages')\
    .count()\
    .orderBy(desc('count'))\
    .withColumn('proportion', round(col('count') / count_languages,3))\
    .withColumn('cumulative_count', sum(col('count')).over(w))\
    .withColumn("cumulative_proportion", round(col("cumulative_count")/count_languages, 2))


/databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/expressions.py:1017: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
# Displaying the top 20 languages
display(df_language_count.limit(20))

/databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/expressions.py:1017: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


languages,count,proportion,cumulative_count,cumulative_proportion
English,55120,0.272,55120,0.27
German,14021,0.069,69141,0.34
French,13427,0.066,82568,0.41
Russian,12922,0.064,95490,0.47
Simplified Chinese,12782,0.063,108272,0.54
Spanish - Spain,12234,0.06,120506,0.6
Japanese,10368,0.051,130874,0.65
Italian,9305,0.046,140179,0.69
Portuguese - Brazil,6750,0.033,146929,0.73
Korean,6601,0.033,153530,0.76


Databricks visualization. Run in Databricks to view.

The top 5 languages are English, German, French, Russian, simplified Chinese, Spanish.

English stands far ahead of the other languages: English stands for 27% of the languages of the video games whereas the subsequent ones (German, French, Russian, Simplified Chinese and Spanish) have a share of around 6 to 7%.

The top 10 languages represent 3/4 of the total languages.


## Are there many games prohibited for children under 16/18?

In [0]:
df_age = df\
    .select('appid', 'name', 'required_age')\
    .dropDuplicates()

# Descriptive statistics of required_age
df_age.describe(['required_age']).show()

# Displaying the modalities taken by required_age
print("Values taken by required_age:")
display(df_age.select(col('required_age')).distinct())

+-------+------------------+
|summary|      required_age|
+-------+------------------+
|  count|             55691|
|   mean|0.1978882344490734|
| stddev| 2.296292461481828|
|    min|                 0|
|    max|            MA 15+|
+-------+------------------+

Values taken by required_age:


required_age
0
14
21+
9
16
35
6
12
7
7+


We need to change the `required_age` variable type from string to integer: 
- extract numbers in this column (in particular, this will extract the number 15 when the required age is "MA 15+");
- convert the result into integer using `cast` method.

In [0]:
# Extract sequences of digits from required_age
df = df.withColumn(
    'required_age',
    regexp_extract(col('required_age'), r'(\d+)', 1)
)

# Convert to integer
df = df.withColumn('required_age', col('required_age').cast('integer'))

# Displaying the modalities taken by required_age
display(df.select(col('required_age')).distinct())

required_age
15
20
21
17
180
13
8
6
35
3


In [0]:
# Looking at rows where required age equals 180
display(df.filter(col('required_age')==180))

appid,categories,ccu,developer,discount,genre,header_image,initialprice,languages,name,negative,owners,positive,price,publisher,release_date,required_age,short_description,tags,type,website,release_year,owners_min,owners_max,owners_midpoint,revenue,total_reviews,score,platform_linux,platform_mac,platform_windows
1091210,"List(Single-player, Steam Achievements)",0,ТЯН ТЯН ТЯН ЛАМПОВАЯ ТЯН INDUSTRIES,0.0,Action,https://cdn.akamai.steamstatic.com/steam/apps/1091210/header.jpg?t=1582325407,0.99,English,Kissing Simulator,25,"0 .. 20,000",62,0.99,Kavkaz Sila Games,2019/07/15,180,Kissing Simulator,"List(null, null, null, null, null, null, null, 12, null, null, null, null, null, null, null, null, null, 30, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 9, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 10, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 10, null, null, null, null, null, null, null, null, null, null, null, null, 11, null, null, null, null, null, 10, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 13, null, null, null, 22, null, null, null, null, null, null, null, null, 10, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 12, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 19, null, null, null, null, null, null, 31, 11, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 11, null, null, null, null, null, null, null, null, null, null, null, null, null, null)",game,,2019,0,20000,10000.0,9900.0,87,0.7126436781609196,false,false,true
1091210,"List(Single-player, Steam Achievements)",0,ТЯН ТЯН ТЯН ЛАМПОВАЯ ТЯН INDUSTRIES,0.0,Simulation,https://cdn.akamai.steamstatic.com/steam/apps/1091210/header.jpg?t=1582325407,0.99,English,Kissing Simulator,25,"0 .. 20,000",62,0.99,Kavkaz Sila Games,2019/07/15,180,Kissing Simulator,"List(null, null, null, null, null, null, null, 12, null, null, null, null, null, null, null, null, null, 30, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 9, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 

In [0]:
# We suppose the 180 value is a typo and replace it by 18
df = df.replace(180, 18, subset=['required_age'])

In [0]:
# Displaying the modalities taken by required_age
display(df.select(col('required_age')).distinct())

required_age
15
20
21
17
13
8
6
35
3
16


In [0]:
df_age = df\
    .select('appid', 'name', 'required_age')\
    .dropDuplicates()\
    .groupBy('required_age')\
    .count()\
    .orderBy('required_age')

display(df_age)

required_age,count
0,55030
3,3
5,1
6,4
7,3
8,3
9,1
10,7
12,32
13,26


Databricks visualization. Run in Databricks to view.

In [0]:
# Create the age category
df = df.withColumn(
    'age_category',
    when(col('required_age') < 16, '0-15')
    .when((col('required_age') >= 16) & (col('required_age') < 18), '16-17')
    .otherwise("18+")
)

In [0]:
df_age_cat = df\
    .select('appid', 'name', 'age_category')\
    .dropDuplicates()\
    .groupBy('age_category')\
    .count()

display(df_age_cat)

age_category,count
0-15,55385
18+,230
16-17,76


Databricks visualization. Run in Databricks to view.

In [0]:
df_age_nodup = df\
    .select('appid', 'name', 'required_age')\
    .dropDuplicates()

# Proportion of games prohibited for children under 16, under 18
proportion_sup16 = builtins.round(df_age_nodup.filter(col('required_age') >= 16).count()/nb_game * 100, 2)
proportion_sup18 = builtins.round(df_age_nodup.filter(col('required_age') >= 18).count()/nb_game * 100, 2)

print(f"{proportion_sup16}% of the games are prohibited for children under 16 ({df_age_nodup.filter(col('required_age') >= 16).count()} video games).")
print(f"{proportion_sup18}% of the games are prohibited for children under 18 ({df_age_nodup.filter(col('required_age') >= 18).count()} video games).")

0.55% of the games are prohibited for children under 16 (306 video games).
0.41% of the games are prohibited for children under 18 (230 video games).


The huge majority of the video games is allowed for children under 16.  
Only 0.55% is prohibited for children under 16 (306 video games). 0.41% is prohibited for children under 18 (230 video games).

### Genre analysis


#### What are the most represented genres?

In [0]:
df_genre = df\
    .select('appid', 'genre')\
    .dropDuplicates()

In [0]:
count_genre = df_genre.count()
w = Window.orderBy(desc("count")) # define a window, in order to compute the cumulative count and proportion

df_genre_count = df_genre\
    .groupBy('genre')\
    .count()\
    .orderBy(desc('count'))\
    .withColumn('proportion', round(col('count') / count_genre, 3))\
    .withColumn('cumulative_count', sum(col('count')).over(w))\
    .withColumn("cumulative_proportion", round(col("cumulative_count")/count_genre, 2))

display(df_genre_count.limit(10))

/databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/expressions.py:1017: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


genre,count,proportion,cumulative_count,cumulative_proportion
Indie,39681,0.252,39681,0.25
Action,23759,0.151,63440,0.4
Casual,22086,0.14,85526,0.54
Adventure,21431,0.136,106957,0.68
Strategy,10895,0.069,117852,0.75
Simulation,10836,0.069,128688,0.82
RPG,9534,0.061,138222,0.88
Early Access,6145,0.039,144367,0.92
Free to Play,3393,0.022,147760,0.94
Sports,2666,0.017,150426,0.96


Databricks visualization. Run in Databricks to view.

The top 5 genres are indie, action, casual, adventure, strategy: they account for 75% of the genres in the dataset.  
Indie is the dominant genre with a market share of 25%. Action, casual and adventure follow with around 14-15% of the market.

#### Are there any genres that have a better positive/negative review ratio?

We will look for genres which display a high proportion of positive reviews.

In [0]:
df_genre_score = df\
    .filter(col('total_reviews') > 0)\
    .select('appid', 'name', 'genre', 'score')\
    .dropDuplicates()\
    .groupBy('genre')\
    .agg(
        round(mean("score"), 3).alias("avg_score"),
        count("*").alias("nb_games")
    )\
    .orderBy(desc('avg_score'))

In [0]:
display(df_genre_score)

genre,avg_score,nb_games
Casual,0.743,21990
Indie,0.742,39558
Adventure,0.739,21397
Game Development,0.738,159
RPG,0.731,9520
Action,0.73,23695
Accounting,0.723,16
Free to Play,0.722,3388
Design & Illustration,0.721,405
Web Publishing,0.72,89


Databricks visualization. Run in Databricks to view.

Casual, indie and adventure are the top 3 genres with the highest proportion of positive reviews.

#### Do some publishers have favorite genres?

We determine the most frequent genre per publisher.

In [0]:
# Number of video games by publisher
df_publisher = df\
    .select('appid', 'name', 'publisher')\
    .dropDuplicates()\
    .groupBy('publisher')\
    .count()\
    .withColumnRenamed('count', 'count_publisher')

display(df_publisher.limit(5))

publisher,count_publisher
Kedronic UAB,5
Aviahel group,1
Atari,40
"Headup, WhisperGames",5
"9 Eyes Game Studio, Stately Snail",3


In [0]:
# Number of video games by publisher and genre
df_genre_publisher = df\
    .select('appid', 'name', 'publisher', 'genre')\
    .dropDuplicates()\
    .groupBy('publisher', 'genre')\
    .count()\
    .withColumnRenamed('count', 'count_genre_publisher')

In [0]:
# Merge the dataframes
df_genre_publisher_def = df_genre_publisher.join(
    df_publisher,
    on=['publisher'],
    how="left" 
)

# By publisher, compute the proportion of each genre
df_genre_publisher_def = df_genre_publisher_def.withColumn(
    'proportion_genre', 
    col('count_genre_publisher')/col('count_publisher')*100
    )

In [0]:
# Definition of the Window
w = Window.partitionBy('publisher').orderBy(desc('proportion_genre')) 

# Add a row number by group of publisher
df_genre_publisher_ranked = df_genre_publisher_def.withColumn("row_num", row_number().over(w))

# Keep the first occurence for each publisher: the best genre
df_genre_publisher_first = df_genre_publisher_ranked.filter(col("row_num") == 1).drop("row_num").orderBy(desc(col('proportion_genre')), desc(col('count_publisher')))

In [0]:
# Displaying publishers with the highest proportion, ordered by decreasing number of video games
display(df_genre_publisher_first\
    .orderBy(desc('proportion_genre'), desc('count_publisher'))\
    .limit(10))

publisher,genre,count_genre_publisher,count_publisher,proportion_genre
8floor,Casual,202,202,100.0
HH-Games,Casual,132,132,100.0
Hosted Games,RPG,79,79,100.0
Boogygames Studios,Indie,78,78,100.0
Tero Lunkka,Indie,68,68,100.0
RewindApp,Indie,59,59,100.0
Ripknot Systems,Casual,55,55,100.0
Hede,Indie,54,54,100.0
Ghost_RUS Games,Indie,50,50,100.0
HexWar Games,Strategy,42,42,100.0


Some publishers have favorite genres. For instance, 8floor, HH-Games and Ripknot Systems have released all of their video games in the 'casual' genre.

#### What are the most lucrative genres?

In [0]:
df_revenue = df\
    .select('appid', 'genre', 'revenue', 'owners_midpoint')\
    .dropDuplicates()\
    .groupBy('genre')\
    .agg(
        sum('revenue').alias('sum_revenue'),
        sum('owners_midpoint').alias('sum_units_sold')
        )\
    .withColumn('avg_revenue_per_unit', round(col('sum_revenue')/col('sum_units_sold'), 2))

In [0]:
display(df_revenue.orderBy(desc('sum_revenue')).limit(5))

genre,sum_revenue,sum_units_sold,avg_revenue_per_unit
Action,5.87564541E10,4.77781E9,12.3
Adventure,3.724573845E10,2.65988E9,14.0
Indie,3.23465771E10,3.18869E9,10.14
RPG,2.71731431E10,1.746625E9,15.56
Strategy,2.015004105E10,1.636135E9,12.32


In [0]:
display(df_revenue.orderBy(desc('avg_revenue_per_unit')).limit(5))

genre,sum_revenue,sum_units_sold,avg_revenue_per_unit
Web Publishing,1.0477075E8,5355000.0,19.57
Audio Production,1.138033E8,6325000.0,17.99
RPG,2.71731431E10,1.746625E9,15.56
Racing,2.723481E9,1.923E8,14.16
Simulation,1.876974955E10,1.3336E9,14.07


Based on the total revenue of the period, the top 3 most lucrative genres are action, adventure and indie.  
If we choose another criteria, such as the revenue per video game, the rank is different: web publishing, audio production and RPG.

### Platform analysis 

#### Are most games available on Windows/Mac/Linux? 

In [0]:
df_platform = df\
    .select('appid', 'name', 'platform_linux', 'platform_mac', 'platform_windows')\
    .dropDuplicates()


df_platform_count = df_platform.agg(
    sum(col("platform_linux").cast("int")).alias("linux"),
    sum(col("platform_mac").cast("int")).alias("mac"),
    sum(col("platform_windows").cast("int")).alias("windows"),
)

In [0]:
display(df_platform_count.toPandas().melt(var_name='platform', value_name='count'))

platform,count
linux,8458
mac,12770
windows,55676


Databricks visualization. Run in Databricks to view.

Windows is the dominant platform with a share of 72% of the video games, and Mac and Linux follow way behind with respectively 17% and 11% of the market.

#### Do certain genres tend to be preferentially available on certain platforms?
For each genre, we compute the number of video games available on each platform.

In [0]:
df_platform_genre = df\
    .select('appid', 'name', 'genre', 'platform_linux', 'platform_mac', 'platform_windows')\
    .dropDuplicates()

In [0]:
df_platform_genre_count = df_platform_genre\
    .groupBy('genre')\
    .agg(
        sum(col("platform_linux").cast("int")).alias("linux"),
        sum(col("platform_mac").cast("int")).alias("mac"),
        sum(col("platform_windows").cast("int")).alias("windows")
)

In [0]:
display(df_platform_genre_count.toPandas().melt(
    id_vars='genre', 
    var_name='platform', 
    value_name='count')
)

genre,platform,count
Adventure,linux,3302
Violent,linux,22
Racing,linux,304
Nudity,linux,7
Indie,linux,6978
Strategy,linux,1826
Education,linux,19
Casual,linux,3305
RPG,linux,1524
Simulation,linux,1532


Databricks visualization. Run in Databricks to view.

All genres are mostly on Windows platform. 
We note that the movie genre is exclusively found on Windows platform.
Audio production, early access, education, photo editing, software training, utilities, video production have a relatively high proportion of video games on Windows (more than 80%).